In [ ]:
# import required modules 
    # copied these over from part 1
from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache
from pathlib import Path
import anndata
import numpy as np
import os
import scanpy as sc
import pandas as pd
import time
import matplotlib.pyplot as plt

In [ ]:
# assigning our own broad cluster and subcluster hierarchy

# LOAD IN DOWNSAMPLED, MERGED OBJECT WITH CLUSTER METADATA ATTACHED 
adata = anndata.read_h5ad("/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data/downsampled_objs/20260121_WMB_10xv3_downsampled_and_metadata.h5ad")
adata



# list clusters from object metadata
adata.obs["cluster"]

# create empty metadata columns 
adata.obs["Broad_Cluster"] = pd.NA
adata.obs["Subcluster"] = pd.NA

# CREATE DICTIONARY FOR BROAD CLUSTERS (how you want the broad clusters organized)

# very detailed clusters important to the research question (aka immune, vascular, etc. for ARIA project) 
Broad_Cluster_map_by_clusters = {
    "Microglia": ["5312 Microglia NN_1"], 
    "BAM" : ["5313 BAM NN_1", "5314 BAM NN_1"], 
    "Monocytes" : ["5315 Monocytes NN_1"], 
    "Dendritic Cells" : ["5316 DC NN_1", "5317 DC NN_1", "5318 DC NN_1"], 
    "B cells" : ["5319 B cells NN_1"], 
    "ILC" : ["5320 ILC NN_2"], 
    "NK cells" : ["5321 NK cells NN_3"], 
    "T cells" : ["5322 T cells NN_4"], 
    "ABC" : ["5293 ABC NN_1", "5294 ABC NN_1", "5295 ABC NN_1"], 
    "Pericytes_1" : ["5304 Peri NN_1"], 
    "Pericytes_2" : ["5305 Peri NN_1"], 
    "SMC_1" : ["5306 SMC NN_1"], 
    "SMC_2" : ["5307 SMC NN_1"], 
    "SMC_3" : ["5308 SMC NN_1"], 
    "Endo_1" : ["5309 Endo NN_1"], 
    "Endo_2" : ["5310 Endo NN_1"],
    "Endo_3" : ["5311 Endo NN_1"],
    "COP" : ["5272 COP NN_1", "5273 COP NN_1", "5274 COP NN_1", "5275 COP NN_1", "5276 COP NN_1", "5277 COP NN_1"], 
    "NFOL" : ["5278 NFOL NN_2", "5279 NFOL NN_2", "5280 NFOL NN_2", "5281 NFOL NN_2"], 
    "MFOL" : ["5282 MFOL NN_3", "5283 MFOL NN_3"], 
    "MOL" : ["5284 MOL NN_4", "5285 MOL NN_4", "5286 MOL NN_4", "5287 MOL NN_4", "5288 MOL NN_4"], 
    "Bergmann Glia" : ["5206 Bergmann NN_1"]
}

Broad_Cluster_map_by_supertype = {
    "VLMC_1" : ["1187 VLMC NN_1"], 
    "VLMC_2" : ["1188 VLMC NN_2"], 
    "VLMC_3" : ["1189 VLMC NN_3"], 
    "VLMC_4" : ["1190 VLMC NN_4"], 
    "OPC" : ["1179 OPC NN_1", "1180 OPC NN_2"],
    "Ependymal_1" : ["1175 Ependymal NN_1"], 
    "Ependymal_2" : ["1176 Ependymal NN_2"], 
    "Hypendymal" : ["1177 Hypendymal NN_1"], 
    "CP" : ["1178 CHOR NN_1"]
    
}


# less detailed clusters (ARIA project: we don't need astrocytes annotated by region)
Broad_Cluster_map_by_subclass = {
    "Astrocytes" : ["317 Astro-CB NN", "318 Astro-NT NN", "319 Astro-TE NN", "320 Astro-OLF NN"], 
    "Astroependymal" : ["321 Astroependymal NN"], 
    "Tanycyte" : ["322 Tanycyte NN"], 
    "OEC" : ["328 OEC NN"] 
}

# very general clusters for neuronal classes (not important to ARIA project) 
Broad_Cluster_map_by_class = {
    "Dopaminergic_Neurons" : ["21 MB Dopa"],
    "Serotonergic_Neurons" : ["22 MB-HB Sero"],
    "GABAergic_Neurons" : ["05 OB-IMN GABA", "06 CTX-CGE GABA", "07 CTX-MGE GABA", "08 CNU-MGE GABA", "09 CNU-LGE GABA", "10 LSX GABA", "11 CNU-HYa GABA", "12 HY GABA", "20 MB GABA", "26 P GABA", "27 MY GABA", "28 CB GABA"],
    "Glutamatergic_Neurons" : ["01 IT-ET Glut", "02 NP-CT-L6b Glut", "03 OB-CR Glut", "04 DG-IMN Glut", "13 CNU-HYa Glut", "14 HY Glut", "15 HY Gnrh1 Glut", "16 HY MM Glut", "17 MH-LH Glut", "18 TH Glut", "19 MB Glut", "23 P Glut", "24 MY Glut", "25 Pineal Glut", "29 CB Glut"]
}


# CREATE DICTIONARY FOR SUBCLUSTERS 

Subcluster_map_by_clusters = {
    "Microglia_01": ["5312 Microglia NN_1"], 
    "Monocytes_01" : ["5315 Monocytes NN_1"], 
    "B cells_01" : ["5319 B cells NN_1"],
    "ILC_01" : ["5320 ILC NN_2"], 
    "NK cells_01" : ["5321 NK cells NN_3"], 
    "T cells_01" : ["5322 T cells NN_4"], 
    "Pericytes_01" : ["5304 Peri NN_1"], 
    "Pericytes_02" : ["5305 Peri NN_1"], 
    "SMC_01" : ["5306 SMC NN_1"], 
    "SMC_02" : ["5307 SMC NN_1"], 
    "SMC_03" : ["5308 SMC NN_1"], 
    "Endo_01" : ["5309 Endo NN_1"], 
    "Endo_02" : ["5310 Endo NN_1"],
    "Endo_03" : ["5311 Endo NN_1"],
    "Bergmann Glia_01" : ["5206 Bergmann NN_1"], 
    "BAM_01" : ["5313 BAM NN_1"], 
    "BAM_02" : ["5314 BAM NN_1"], 
    "Dendritic Cells_01" : ["5316 DC NN_1"],   
    "Dendritic Cells_02" : ["5317 DC NN_1"],
    "Dendritic Cells_03" : ["5318 DC NN_1"],
    "ABC_01" : ["5293 ABC NN_1"],    
    "ABC_02" : ["5294 ABC NN_1"],
    "ABC_03" : ["5295 ABC NN_1"],
    "COP_01" : ["5272 COP NN_1"],     
    "COP_02" : ["5273 COP NN_1"],
    "COP_03" : ["5274 COP NN_1"],
    "COP_04" : ["5275 COP NN_1"],
    "COP_05" : ["5276 COP NN_1"],
    "COP_06" : ["5277 COP NN_1"],
    "NFOL_01" : ["5278 NFOL NN_2"],    
    "NFOL_02" : ["5279 NFOL NN_2"],
    "NFOL_03" : ["5280 NFOL NN_2"],
    "NFOL_04" : ["5281 NFOL NN_2"],
    "MFOL_01" : ["5282 MFOL NN_3"], 
    "MFOL_02" : ["5283 MFOL NN_3"], 
    "MOL_01" : ["5284 MOL NN_4"],     
    "MOL_02" : ["5285 MOL NN_4"],
    "MOL_03" : ["5286 MOL NN_4"],
    "MOL_04" : ["5287 MOL NN_4"],
    "MOL_05" : ["5288 MOL NN_4"]
       
}

Subcluster_map_by_supertype = {
    "VLMC_01" : ["1187 VLMC NN_1"], 
    "VLMC_02" : ["1188 VLMC NN_2"], 
    "VLMC_03" : ["1189 VLMC NN_3"], 
    "VLMC_04" : ["1190 VLMC NN_4"], 
    "Ependymal_01" : ["1175 Ependymal NN_1"], 
    "Ependymal_02" : ["1176 Ependymal NN_2"], 
    "Hypendymal_01" : ["1177 Hypendymal NN_1"], 
    "CP_01" : ["1178 CHOR NN_1"],
    "OPC_01" : ["1179 OPC NN_1"],  
    "OPC_02" : ["1180 OPC NN_2"]
    
}

Subcluster_map_by_subclass = {
    "CB_Astrocytes" : ["317 Astro-CB NN"],    
    "NT_Astrocytes" : ["318 Astro-NT NN"],
    "TE_Astrocytes" : ["319 Astro-TE NN"],
    "OLF_Astrocytes" : ["320 Astro-OLF NN"], 
    "Astroependymal_01" : ["321 Astroependymal NN"], 
    "Tanycyte_01" : ["322 Tanycyte NN"], 
    "OEC_01" : ["328 OEC NN"], 
    "Dopaminergic_Neurons_01" : ["215 SNc-VTA-Ramb Foxa1 Dopa"], 
    "Serotonergic_Neurons_01" : ["216 MB-MY Tph2 Glut-Sero"]
 
}

GABA_Subcluster_map_by_subclass = {
    "LAMP5" : ["049 Lamp5 Gaba", "050 Lamp5 Lhx6 Gaba"], 
    "PVALB" : ["051 Pvalb chandelier Gaba", "052 Pvalb Gaba"], 
    "SNCG" : ["047 Sncg Gaba"], 
    "SST" : ["053 Sst Gaba", "056 Sst Chodl Gaba"], 
    "VIP" : ["046 Vip Gaba"]
}

Glut_Subcluster_map_by_subclass = {
    "L2_3_IT" : ["007 L2/3 IT CTX Glut", "008 L2/3 IT ENT Glut", "009 L2/3 IT PIR-ENTl Glut", "019 L2/3 IT PPP Glut", "020 L2/3 IT RSP Glut"], 
    "L5_IT" : ["005 L5 IT CTX Glut"], 
    "L6_IT" : ["004 L6 IT CTX Glut"], 
    "L5_ET" : ["022 L5 ET CTX Glut"], 
    "L6_CT" : ["028 L6b/CT ENT Glut", "029 L6b CTX Glut", "030 L6 CT CTX Glut"]
}


    


In [ ]:
# adding map to object 

# extracting metadata from downsampled object 
meta = adata.obs.copy()

'''
Function: used to map predefined "dictionaries" from above to the downsampled 10xv3 object metadata
Input: metadata row (must be a pandas df) that has ABC cluster metadata informatoin attached 
    (contains columns 'class', 'subclass', and 'cluster'"
Output: labels the pre-defined broad cluster that the cell belongs to in the 'Broad_Cluster' column
'''

def map_broad_cluster(row):
    
    # Mapping by class first 
    # .items() returns broad_cluster (broad) and class (class_list) pairs from the 'Broad_Cluster_map_by_class' df
    for broad, class_list in Broad_Cluster_map_by_class.items():

        # for loop goes through each class in the class_list belonging to the broad_cluster value
        for cls in class_list:

            # if the class matches the class listed in the 'class' column from the downsampled object, the 'broad_cluster' value will be added to the 'Broad_Cluster' column
            if cls in row['class']:
                return broad
    
    # if couldn't find a match for class, then use Broad_Cluster_map_by_subclass
    for broad, subclass_list in Broad_Cluster_map_by_subclass.items():

        # for loop goes through each subclass in the subclass_list belonging to the broad_cluster value
        for subcls in subclass_list:
            
            # if the subclass matches, the 'broad_cluster' value will be added 
            if subcls in row['subclass']:
                return broad

    # if couldn't find a match for subclass, then use Broad_Cluster_map_by_supertype
    for broad, supertype_list in Broad_Cluster_map_by_supertype.items():

        # for loop goes through each supertype in the supertype_list belonging to the broad_cluster value
        for suptype in supertype_list:
            
            # if the supertype matches, the 'broad_cluster' value will be added 
            if suptype in row['supertype']:
                return broad
                
    # if couldn't find a match for subclass, then use Broad_Cluster_map_by_clusters
    for broad, cluster_list in Broad_Cluster_map_by_clusters.items():

        # if the cluster matches, the 'broad_cluster' value will be added 
        if row['cluster'] in cluster_list:
            return broad
    
    # if nothing matches
    return "Unknown"

def map_subcluster(row):

    # for neuronal subclusters 
    if row['Broad_Cluster'] == "Glutamatergic_Neurons":
        for sub, subclass_list in Glut_Subcluster_map_by_subclass.items():
            for subcls in subclass_list:
                if subcls in row['subclass']:
                    return sub
                    
        return None 
    
    if row['Broad_Cluster'] == "GABAergic_Neurons":
        for sub, subclass_list in GABA_Subcluster_map_by_subclass.items():
            for subcls in subclass_list:
                if subcls in row['subclass']:
                    return sub
                    
        return None    

    
    # nonneuronal subclusters  

    # mapping by subclass dictionary 
    for sub, subclass_list in Subcluster_map_by_subclass.items():
        for s in subclass_list:
            if s in row['subclass']:
                return sub
                
    # mapping by supertype dictionary  
    for sub, supertype_list in Subcluster_map_by_supertype.items():
        for su in supertype_list:
            if su in row['supertype']:
                return sub

    # mapping by cluster dictionary
    for sub, cluster_list in Subcluster_map_by_clusters.items():
        for cls in cluster_list:
            if cls in row['cluster']:
                return sub
        
    return None




# applying the function to the metadata object 
meta['Broad_Cluster'] = meta.apply(map_broad_cluster, axis = 1)
meta['Subcluster'] = meta.apply(map_subcluster, axis = 1)



# assign back to the downsampled obj
adata.obs = meta




# SAVE OBJECT
# save downsampled object with cluster metadata and our own cluster levels added in
adata.write_h5ad("/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data/downsampled_objs/20260121_WMB_10xv3_FINAL.h5ad")



In [ ]:
adata.obs["Subcluster"].value_counts()
